In [16]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score



In [17]:
default = pd.read_csv('creditcard.csv')
default.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,...,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,...,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,...,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,...,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,...,1814,0,0,1000,664,1500,0,0,0,0


In [18]:
report = sv.analyze(default)
report.show_html('report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


## EDA Highlights

The dataset has 24,000 rows, 24 features, no missing values, and 22 duplicates . Financial variables (bill and payment amounts) are highly skewed with large outliers, while most customers show on-time or slightly early payments. Credit limits vary widely, and the average age is mid-30s. Overall, the data is clean but will likely need scaling and handling of skewness before modeling.

In [19]:
# Prepare features and target
target_col = 'default payment next month'
X = default.drop(target_col, axis=1)
y = default[target_col]

# Split data with stratification because the target is imbalanced
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# RandomForest tuning grid
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
print('RF best params:', rf_grid.best_params_)
print('RF best CV accuracy:', rf_grid.best_score_)

# XGBoost tuning grid
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.01],
    'subsample': [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    xgb_param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

xgb_grid.fit(X_train, y_train)
xgb_best = xgb_grid.best_estimator_
print('XGB best params:', xgb_grid.best_params_)
print('XGB best CV accuracy:', xgb_grid.best_score_)

# Evaluate tuned models on the holdout test set
for name, model in [('RandomForest', rf_best), ('XGBoost', xgb_best)]:
    y_pred = model.predict(X_test)
    print(f"\n{name} test accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))

# Store tuned models for further analysis
rf_tuned = rf_best
xgb_tuned = xgb_best

Fitting 3 folds for each of 24 candidates, totalling 72 fits
RF best params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
RF best CV accuracy: 0.8180729166666668
Fitting 3 folds for each of 16 candidates, totalling 48 fits


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:06:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:06:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:06:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:06:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encode

XGB best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
XGB best CV accuracy: 0.8213541666666666

RandomForest test accuracy: 0.8192
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      3738
           1       0.67      0.36      0.47      1062

    accuracy                           0.82      4800
   macro avg       0.75      0.66      0.68      4800
weighted avg       0.80      0.82      0.80      4800


XGBoost test accuracy: 0.8190
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.66      0.38      0.48      1062

    accuracy                           0.82      4800
   macro avg       0.75      0.66      0.69      4800
weighted avg       0.80      0.82      0.80      4800



In [20]:
# Evaluate tuned models on entire dataset using 5-fold CV
rf_cv_scores = cross_val_score(rf_tuned, X, y, cv=5, scoring='accuracy')
xgb_cv_scores = cross_val_score(xgb_tuned, X, y, cv=5, scoring='accuracy')

print('RandomForest CV Accuracy Scores:', rf_cv_scores)
print('Mean:', rf_cv_scores.mean(), 'Std:', rf_cv_scores.std())

print('XGBoost CV Accuracy Scores:', xgb_cv_scores)
print('Mean:', xgb_cv_scores.mean(), 'Std:', xgb_cv_scores.std())

/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:07:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:07:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:07:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:07:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encode

RandomForest CV Accuracy Scores: [0.81791667 0.82083333 0.818125   0.81708333 0.825     ]
Mean: 0.8197916666666668 Std: 0.0028927591596182756
XGBoost CV Accuracy Scores: [0.81979167 0.82229167 0.823125   0.81791667 0.82395833]
Mean: 0.8214166666666667 Std: 0.002237620263682935


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:07:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## Model Performance

Both models achieved ~82% CV accuracy with low standard deviation (~0.002), indicating stable performance across folds. The consistency suggests good generalization without overfitting to specific splits.

## Classification Report Analysis

The reports show high accuracy but poor recall for defaults (~35%), meaning many defaults are missed. This shifts evaluation from accuracy to metrics like recall or F1, crucial for imbalanced credit risk assessment.

In [21]:
# Define baseline models with default parameters
rf = RandomForestClassifier(random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

In [22]:
# repeat baseline model training but use recall as evaluation metric instead of accuracy
rf_cv_recall = cross_val_score(rf, X, y, cv=3, scoring='recall')
xgb_cv_recall = cross_val_score(xgb, X, y, cv=3, scoring='recall')
print('Random Forest CV Recall:', rf_cv_recall)
print('XGBoost CV Recall:', xgb_cv_recall)  
print('Random Forest CV Recall Mean:', rf_cv_recall.mean())
print('XGBoost CV Recall Mean:', xgb_cv_recall.mean()) 

/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:07:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:07:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:07:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Random Forest CV Recall: [0.36630865 0.36101695 0.38248588]
XGBoost CV Recall: [0.36743923 0.37288136 0.37683616]
Random Forest CV Recall Mean: 0.3699371579376562
XGBoost CV Recall Mean: 0.3723855817761213


In [23]:
# Models with class imbalance adjustment
scale_pos = len(y[y==0]) / len(y[y==1])
rf_balanced = RandomForestClassifier(class_weight='balanced', random_state=42)
xgb_balanced = XGBClassifier(scale_pos_weight=scale_pos, use_label_encoder=False, eval_metric='logloss', random_state=42)

rf_bal_cv = cross_val_score(rf_balanced, X, y, cv=5, scoring='recall')
xgb_bal_cv = cross_val_score(xgb_balanced, X, y, cv=5, scoring='recall')

print('RF Balanced CV Recall Mean:', rf_bal_cv.mean(), 'Std:', rf_bal_cv.std())
print('XGB Balanced CV Recall Mean:', xgb_bal_cv.mean(), 'Std:', xgb_bal_cv.std())

/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:08:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:08:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:08:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:08:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encode

RF Balanced CV Recall Mean: 0.3377256647692278 Std: 0.008744369172959643
XGB Balanced CV Recall Mean: 0.5694102319703367 Std: 0.003662397613502133


## Impact of Class Imbalance Adjustment

For RandomForest, recall remained similar at 0.34, while XGBoost improved significantly to 0.57. This shows scale_pos_weight was more effective for XGBoost, boosting minority class detection with low variance.

In [24]:
# XGBoost with subsample=0.8
xgb_sub = XGBClassifier(subsample=0.8, use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_sub_cv = cross_val_score(xgb_sub, X, y, cv=5, scoring='recall')
print('XGB with subsample CV Recall Mean:', xgb_sub_cv.mean(), 'Std:', xgb_sub_cv.std())

/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:08:12] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:08:13] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:08:13] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:08:14] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encode

XGB with subsample CV Recall Mean: 0.3667362453429323 Std: 0.008135401500412786


## Impact of XGBoost Subsampling

Subsample=0.8 improved generalization, slightly increasing recall stability with lower variance compared to full sampling.

In [25]:
# hyperparameter tuning for random forest wiht 2 hyperparameters and 2 values each
rf_param_grid_simple = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
}   

rf_grid_simple = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid_simple, cv=3, scoring='recall')
rf_grid_simple.fit(X_train, y_train)
print('RF simple tuning best params:', rf_grid_simple.best_params_)
print('RF simple tuning best CV recall:', rf_grid_simple.best_score_)

# Brief tuning for XGBoost with 2-3 features
xgb_param_grid_simple = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2]
}

xgb_grid_simple = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42), xgb_param_grid_simple, cv=3, scoring='recall')
xgb_grid_simple.fit(X_train, y_train)
print('XGB simple tuning best params:', xgb_grid_simple.best_params_)
print('XGB simple tuning best CV recall:', xgb_grid_simple.best_score_)


RF simple tuning best params: {'max_depth': None, 'n_estimators': 200}
RF simple tuning best CV recall: 0.3673144876325088


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:09:45] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:09:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:09:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:09:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encode

XGB simple tuning best params: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200}
XGB simple tuning best CV recall: 0.37414405781477705


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [11:09:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## Tuning Results

Simple tuning did not significantly improve recall for both models (see the printed best CV recall values above, which may be similar to baseline). I chose n_estimators, max_depth, learning_rate as they control model capacity and learning rate. These parameters may not have been the most impactful for this dataset.